# 03 - Feature Engineering

Turning the cleaned weekly counts into features a model can learn from.

The setup: **standing at week t, predict week t+1** for each district. So every feature only uses week t and earlier, and the targets come from week t+1.

Two targets, so either kind of model can be trained from the same file:
- `target_log_next` - next week's cases on the log scale (regression)
- `risk_next` - next week's risk level: low / medium / high (classification)

Models will be XGBoost / LightGBM (trees), so no scaling is needed.

**Input:** `dataset/processed/02_cleaned_weekly_data.csv`
**Output:** `dataset/processed/03_features.csv` and `dataset/processed/03_risk_thresholds.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../dataset")
IN_PATH = DATA_DIR / "processed" / "02_cleaned_weekly_data.csv"
OUT_PATH = DATA_DIR / "processed" / "03_features.csv"
THRESH_PATH = DATA_DIR / "processed" / "03_risk_thresholds.csv"

TRAIN_END = "2023-12-31"  # train on 2007-2023, test on 2024-2026

In [2]:
df = pd.read_csv(IN_PATH, parse_dates=["start_date", "end_date"])
df = df.sort_values(["district", "start_date"]).reset_index(drop=True)

df["split"] = np.where(df["start_date"] <= TRAIN_END, "train", "test")

print(df.shape)
df["split"].value_counts()

(26728, 9)


split
train    23114
test      3614
Name: count, dtype: int64

## 1. Log of cases

Counts are very skewed (lots of zeros, outbreaks in the thousands), so I work with `log1p(cases)`. This stops a few huge outbreak weeks from dominating the training.

In [3]:
df["log_cases"] = np.log1p(df["cases"])
df[["cases", "log_cases"]].describe().round(2)

,cases,log_cases
count,26728.00,26728.00
mean,33.63,2.35
std,86.86,1.54
min,0.00,0.00
25%,2.00,1.10
50%,9.00,2.30
75%,30.00,3.43
max,2631.00,7.88


## 2. Past weeks (lags)

What happened in the last few weeks is the strongest signal for next week. I use the last 3 weeks, plus the same week last year (52 weeks ago) for the yearly pattern. All lags are taken within each district.

In [4]:
g = df.groupby("district")["log_cases"]

for k in [1, 2, 3, 52]:
    df[f"log_lag{k}"] = g.shift(k)

colombo = df[df["district"] == "Colombo"].set_index("start_date")
colombo[["cases", "log_cases", "log_lag1", "log_lag2", "log_lag52"]].iloc[52:57].round(2)

,cases,log_cases,log_lag1,log_lag2,log_lag52
start_date,,,,,
2007-12-22,40,3.71,3.81,4.39,4.28
2007-12-29,49,3.91,3.71,3.81,3.71
2008-01-05,44,3.81,3.91,3.71,3.78
2008-01-12,23,3.18,3.81,3.91,3.66
2008-01-19,38,3.66,3.18,3.81,3.97


## 3. Recent average and trend

- `roll4_mean` / `roll8_mean`: average of the last 4 / 8 weeks (including this week). Smoother than single weeks.
- `change_1w`: this week minus last week. Positive = cases are rising.

In [5]:
df["roll4_mean"] = g.transform(lambda s: s.rolling(4).mean())
df["roll8_mean"] = g.transform(lambda s: s.rolling(8).mean())
df["change_1w"] = df["log_cases"] - df["log_lag1"]

## 4. Season

Dengue follows the monsoons, so the time of year matters. Week of year is turned into sin/cos so that week 52 and week 1 end up close together instead of 51 apart.

In [6]:
week_of_year = df["start_date"].dt.isocalendar().week.astype(int)
df["week_sin"] = np.sin(2 * np.pi * week_of_year / 52)
df["week_cos"] = np.cos(2 * np.pi * week_of_year / 52)

## 5. District

LightGBM can use `district` directly as a category. For XGBoost I also add a simple number code.

In [7]:
df["district"] = df["district"].astype("category")
df["district_code"] = df["district"].cat.codes
dict(enumerate(df["district"].cat.categories))

{0: 'Ampara',
 1: 'Anuradhapura',
 2: 'Badulla',
 3: 'Batticaloa',
 4: 'Colombo',
 5: 'Galle',
 6: 'Gampaha',
 7: 'Hambanthota',
 8: 'Jaffna',
 9: 'Kalmune',
 10: 'Kalutara',
 11: 'Kandy',
 12: 'Kegalle',
 13: 'Kilinochchi',
 14: 'Kurunegala',
 15: 'Mannar',
 16: 'Matale',
 17: 'Matara',
 18: 'Monaragala',
 19: 'Mullaitivu',
 20: 'NuwaraEliya',
 21: 'Polonnaruwa',
 22: 'Puttalam',
 23: 'Ratnapura',
 24: 'Trincomalee',
 25: 'Vavuniya'}

## 6. Targets

**Regression:** `target_log_next` = next week's `log_cases` (turn predictions back into cases with `np.expm1`).

**Classification:** a big district like Colombo has hundreds of cases every week, while Mullaitivu usually has 0-2, so one national cut-off doesn't work. Instead each district gets its own thresholds from its **training years only** (so the test period doesn't leak in):
- **low:** up to the district's 75th percentile
- **medium:** above the 75th, up to the 90th percentile
- **high:** above the 90th percentile

I also keep `target_imputed`, which says if next week's value was estimated in cleaning, so those weeks can be left out when scoring the model.

In [8]:
df["cases_next"] = df.groupby("district", observed=True)["cases"].shift(-1)
df["target_log_next"] = g.shift(-1)
df["target_imputed"] = df.groupby("district", observed=True)["is_imputed"].shift(-1)

thresholds = (
    df[df["split"] == "train"]
    .groupby("district", observed=True)["cases"]
    .quantile([0.75, 0.90])
    .unstack()
    .set_axis(["medium_above", "high_above"], axis=1)
)
thresholds.sort_values("high_above")

,medium_above,high_above
district,,
Mullaitivu,2.0,5.0
Kilinochchi,2.0,6.0
Ampara,4.0,9.0
Mannar,3.0,10.0
NuwaraEliya,6.0,11.0
Polonnaruwa,8.0,14.0
Vavuniya,5.0,15.0
Monaragala,9.0,20.0
Anuradhapura,13.0,26.0


In [9]:
t = df[["district"]].join(thresholds, on="district")

df["risk_next"] = np.select(
    [df["cases_next"] > t["high_above"], df["cases_next"] > t["medium_above"]],
    ["high", "medium"],
    default="low",
)
df["risk_next"] = df["risk_next"].where(df["cases_next"].notna())

pd.crosstab(df["split"], df["risk_next"], normalize="index").round(2)

risk_next,high,low,medium
split,,,
test,0.16,0.54,0.30
train,0.10,0.76,0.14


Training comes out at about 76% low / 14% medium / 10% high, as expected from the percentiles. The test years (2024-2026) have far more medium and high weeks (30% / 16%) because case levels were higher recently. That's worth remembering when looking at test scores.

## 7. Drop incomplete rows

The first 52 weeks of each district have no `log_lag52`, and the last week has no next week to predict. These rows are dropped.

In [10]:
FEATURES = ["log_cases", "log_lag1", "log_lag2", "log_lag3", "log_lag52",
            "roll4_mean", "roll8_mean", "change_1w", "week_sin", "week_cos", "district_code"]
TARGETS = ["target_log_next", "risk_next"]

before = len(df)
df = df.dropna(subset=FEATURES + TARGETS).reset_index(drop=True)
print("dropped", before - len(df), "rows ->", df.shape)

dropped 1378 rows -> (25350, 24)


## Quick check: how much does each feature relate to next week?

In [11]:
df[FEATURES + ["target_log_next"]].corr()["target_log_next"].drop("target_log_next").sort_values(ascending=False).round(2)

roll4_mean       0.91
log_cases        0.90
roll8_mean       0.89
log_lag1         0.88
log_lag2         0.86
log_lag3         0.84
log_lag52        0.60
change_1w        0.04
week_cos         0.03
week_sin         0.00
district_code   -0.21
Name: target_log_next, dtype: float64

This week and the recent averages are the strongest (~0.9). Last year's value is weaker (0.6) but still useful. Trend, season and district code show almost no *linear* relation, but trees can still use them in combination (e.g. a rise in a certain season). The district code's sign means nothing because the numbering is just alphabetical.

## Checks

In [12]:
assert df[FEATURES + TARGETS].notna().all().all()
assert not df.duplicated(["district", "start_date"]).any()

# the target really is the next week's value
nxt = df.groupby("district", observed=True)["log_cases"].shift(-1)
gap = df.groupby("district", observed=True)["start_date"].shift(-1) - df["start_date"]
one_week = gap == pd.Timedelta(days=7)
assert (df.loc[one_week, "target_log_next"] == nxt[one_week]).all()

print("all checks passed")

all checks passed


## Save

In [13]:
cols = ["start_date", "year", "week", "district", "split", "cases", "is_imputed", "dup_week",
        *FEATURES, "cases_next", "target_log_next", "risk_next", "target_imputed"]

df[cols].sort_values(["start_date", "district"]).to_csv(OUT_PATH, index=False, date_format="%Y-%m-%d")
thresholds.to_csv(THRESH_PATH)
print("saved", len(df), "rows")

saved 25350 rows


## Summary

| Feature | What it is |
|---|---|
| `log_cases` | log1p of this week's cases |
| `log_lag1`-`log_lag3` | same, 1-3 weeks ago |
| `log_lag52` | same week last year |
| `roll4_mean`, `roll8_mean` | average of last 4 / 8 weeks (log scale) |
| `change_1w` | this week minus last week (rising or falling) |
| `week_sin`, `week_cos` | time of year |
| `district` / `district_code` | which district |

**Targets:** `target_log_next` (regression) and `risk_next` (low/medium/high by each district's own 75th/90th percentile from 2007-2023).

The thresholds are saved separately so the dashboard can turn predicted case numbers into the same risk levels.